
# AHS-KT × Junyi 最小可运行 Notebook

这本 Notebook 的目标是：

1. 使用 `/root/autodl-tmp/ahs-kt` 项目代码；
2. 使用 Junyi 原始数据：`data/junyi/junyi_ProblemLog_original.csv`；
3. 在 **不改动项目核心源码** 的前提下，把 `AHS-KT` 的数据构建、配置生成、训练、评估整条链路跑通；
4. 输出 `acc / auc / f1`。

## 先说一个关键现实约束

Junyi 原始日志非常大：大约 **2592 万** 条交互、**24.7 万** 个用户。

如果你的目标是“先把项目跑通”，最短路径不是直接全量训练，而是：
- 仍然使用 Junyi 原始数据；
- 但默认只抽取一个**可控用户子集**来做 quickstart；
- 把完整的数据构建和训练流程验证通过。

因此，这本 Notebook 默认配置是：
- 从满足最少交互数的用户里随机采样 `1000` 个用户；
- 基于这些用户构造 `AHS-KT` 所需 `.npz`；
- 训练一个可跑通版本；
- 给出 `acc / auc / f1`。

如果你后面要跑全量 Junyi，只需要把参数区的 `MAX_USERS` 改成 `None`。


In [1]:

from pathlib import Path
import os
import sys
import json
import time
import random

PROJECT_ROOT = Path('/root/autodl-tmp/ahs-kt')
SRC_ROOT = PROJECT_ROOT / 'src'
RAW_JUNYI_PATH = PROJECT_ROOT / 'data/junyi/junyi_ProblemLog_original.csv'
JUNYI_EXERCISE_PATH = PROJECT_ROOT / 'data/junyi/junyi_Exercise_table.csv'

assert PROJECT_ROOT.exists(), f'找不到项目目录: {PROJECT_ROOT}'
assert RAW_JUNYI_PATH.exists(), f'找不到 Junyi 日志文件: {RAW_JUNYI_PATH}'
assert JUNYI_EXERCISE_PATH.exists(), f'找不到 Junyi 练习元数据文件: {JUNYI_EXERCISE_PATH}'

os.chdir(PROJECT_ROOT)
os.environ.setdefault('OMP_NUM_THREADS', '1')
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

print('PROJECT_ROOT =', PROJECT_ROOT)
print('RAW_JUNYI_PATH =', RAW_JUNYI_PATH)
print('JUNYI_EXERCISE_PATH =', JUNYI_EXERCISE_PATH)
print('OMP_NUM_THREADS =', os.environ['OMP_NUM_THREADS'])


PROJECT_ROOT = /root/autodl-tmp/ahs-kt
RAW_JUNYI_PATH = /root/autodl-tmp/ahs-kt/data/junyi/junyi_ProblemLog_original.csv
JUNYI_EXERCISE_PATH = /root/autodl-tmp/ahs-kt/data/junyi/junyi_Exercise_table.csv
OMP_NUM_THREADS = 1


In [2]:

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from ahskt.config import load_config
from ahskt.data.assist2012 import build_sequence_records, records_to_bundle, shuffle_records
from ahskt.models.ahs_kt import AHSKTModel
from ahskt.training.engine import fit_and_evaluate

print('TensorFlow version =', tf.__version__)
print('GPU devices =', tf.config.list_physical_devices('GPU'))
for gpu_device in tf.config.list_physical_devices('GPU'):
    try:
        tf.config.experimental.set_memory_growth(gpu_device, True)
    except RuntimeError:
        pass


TensorFlow version = 2.8.0
GPU devices = [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]



## 参数区

这几个参数决定 Notebook 是跑“快速验证版”还是“更接近正式实验版”。

默认值含义：
- `MAX_USERS = 1000`：从 Junyi 中抽 1000 个用户，快速把项目跑通；
- `MIN_INTERACTIONS = 5`：只保留至少 5 次交互的用户；
- `SEQUENCE_LENGTH = 100`：与项目现有构建脚本风格一致；
- `EPOCHS = 2`：快速验证训练闭环；
- `N_CLUSTERS = 4`：和现有 assist 构建脚本保持一致。


In [3]:

SEED = 2026
MAX_USERS = 1000         # 改成 None 即可尝试全量 Junyi（会明显更慢）
MIN_INTERACTIONS = 5
SEQUENCE_LENGTH = 100
N_CLUSTERS = 4
BATCH_SIZE = 64
EPOCHS = 2
LEARNING_RATE = 1e-3
PATIENCE = 1
TASK_NAME = 'ahskt_junyi_quickstart_1k'

DATA_DIR = PROJECT_ROOT / 'data/junyi'
OUTPUT_DIR = PROJECT_ROOT / 'outputs/junyi_quick_run'
CONFIG_PATH = PROJECT_ROOT / 'configs/ahskt_junyi_quick.json'
TRAIN_NPZ_PATH = DATA_DIR / 'junyi_quick_train_ahskt.npz'
VALID_NPZ_PATH = DATA_DIR / 'junyi_quick_valid_ahskt.npz'
TEST_NPZ_PATH = DATA_DIR / 'junyi_quick_test_ahskt.npz'
METADATA_PATH = DATA_DIR / 'junyi_quick_metadata.json'

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('TASK_NAME =', TASK_NAME)
print('OUTPUT_DIR =', OUTPUT_DIR)


TASK_NAME = ahskt_junyi_quickstart_1k
OUTPUT_DIR = /root/autodl-tmp/ahs-kt/outputs/junyi_quick_run


In [4]:

# 先只读 user_id，统计 Junyi 全量规模，并决定这次 quickstart 用哪些用户。
user_counts = pd.read_csv(
    RAW_JUNYI_PATH,
    usecols=['user_id'],
    dtype={'user_id': 'int32'},
).groupby('user_id').size()

eligible_users = user_counts[user_counts >= MIN_INTERACTIONS].index.to_numpy(dtype=np.int64)
rng = np.random.default_rng(SEED)

if MAX_USERS is None:
    chosen_users = eligible_users.copy()
else:
    chosen_users = rng.choice(eligible_users, size=min(MAX_USERS, len(eligible_users)), replace=False)

chosen_user_set = set(int(x) for x in chosen_users)

print('Junyi 总交互数 =', int(user_counts.sum()))
print('Junyi 总用户数 =', int(len(user_counts)))
print('满足最少交互数的用户数 =', int(len(eligible_users)))
print('本次实际抽取用户数 =', int(len(chosen_users)))


Junyi 总交互数 = 25925992
Junyi 总用户数 = 247606
满足最少交互数的用户数 = 175359
本次实际抽取用户数 = 1000



## 读取并过滤 Junyi 原始日志

这里采用两阶段策略：
1. 第一阶段只统计用户交互次数；
2. 第二阶段按 chunk 读取原始日志，只保留本次抽到的用户。

这样做的原因很简单：Junyi 太大，没必要一次把全量字符串列都读进内存。


In [5]:

usecols = ['user_id', 'exercise', 'time_done', 'time_taken', 'correct', 'count_attempts', 'count_hints']
dtypes = {
    'user_id': 'int32',
    'exercise': 'category',
    'time_done': 'int64',
    'time_taken': 'float32',
    'correct': 'boolean',
    'count_attempts': 'float32',
    'count_hints': 'float32',
}

chunks = []
for chunk in pd.read_csv(
    RAW_JUNYI_PATH,
    usecols=usecols,
    dtype=dtypes,
    chunksize=1_000_000,
    low_memory=False,
):
    chunk = chunk[chunk['user_id'].isin(chosen_user_set)]
    if not chunk.empty:
        chunks.append(chunk)

log_df = pd.concat(chunks, ignore_index=True)
print('过滤后交互数 =', len(log_df))
print('过滤后用户数 =', log_df['user_id'].nunique())
print('过滤后练习数 =', log_df['exercise'].nunique())
log_df.head()


过滤后交互数 = 144995
过滤后用户数 = 1000
过滤后练习数 = 583


,user_id,exercise,time_done,time_taken,correct,count_attempts,count_hints
0,116991,multiplying_and_dividing_negative_numbers,1411305913182370,6.0,True,1.0,0.0
1,32807,division_4,1417582233310040,10.0,True,1.0,0.0
2,186814,subtraction_2,1393989227330040,8.0,False,3.0,0.0
3,95965,matrix_basic_product,1417696566179890,47.0,True,1.0,0.0
4,68243,radius_and_diameter,1414826871643450,4.0,True,1.0,0.0


In [6]:

exercise_df = pd.read_csv(JUNYI_EXERCISE_PATH, usecols=['name', 'topic', 'area'])
exercise_df['concept_name'] = exercise_df['topic'].fillna(exercise_df['area']).fillna(exercise_df['name'])
exercise_df = exercise_df[['name', 'concept_name']].drop_duplicates('name')

# 这里把 exercise 映射成 question，把 topic/area 映射成 concept。
# 如果某条 exercise 没有 topic，就退化成“exercise 自己就是 concept”。
df = log_df.merge(exercise_df, left_on='exercise', right_on='name', how='left')
df['exercise'] = df['exercise'].astype(str)
df['concept_name'] = df['concept_name'].fillna(df['exercise'])

df['correct'] = df['correct'].fillna(False).astype(np.int32)
df['count_attempts'] = pd.to_numeric(df['count_attempts'], errors='coerce').fillna(1.0).clip(lower=1.0)
df['count_hints'] = pd.to_numeric(df['count_hints'], errors='coerce').fillna(0.0).clip(lower=0.0)
df['time_taken'] = pd.to_numeric(df['time_taken'], errors='coerce').replace([np.inf, -np.inf], np.nan)
median_time = float(df['time_taken'].dropna().median())
if not np.isfinite(median_time) or median_time <= 0:
    median_time = 10.0
df['time_taken'] = df['time_taken'].fillna(median_time).clip(lower=1.0)
df['timestamp'] = pd.to_numeric(df['time_done'], errors='coerce')
df = df[df['timestamp'].notna()].copy()
df['timestamp'] = df['timestamp'].astype(np.int64)

print('清洗后交互数 =', len(df))
print('question 数 =', df['exercise'].nunique())
print('concept 数 =', df['concept_name'].nunique())
df[['user_id', 'exercise', 'concept_name', 'correct', 'time_taken', 'count_attempts', 'count_hints']].head()


清洗后交互数 = 144995
question 数 = 583
concept 数 = 39


,user_id,exercise,concept_name,correct,time_taken,count_attempts,count_hints
0,116991,multiplying_and_dividing_negative_numbers,absolute-value,1,6.0,1.0,0.0
1,32807,division_4,multiplication-division,1,10.0,1.0,0.0
2,186814,subtraction_2,addition-subtraction,0,8.0,3.0,0.0
3,95965,matrix_basic_product,vectors-matrix,1,47.0,1.0,0.0
4,68243,radius_and_diameter,basic-geometry,1,4.0,1.0,0.0


In [7]:

all_users = np.array(sorted(df['user_id'].unique()), dtype=np.int64)
train_user_ids, test_user_ids = train_test_split(all_users, test_size=0.1, random_state=SEED)
train_user_ids, valid_user_ids = train_test_split(train_user_ids, test_size=1/9, random_state=SEED)

train_user_ids = np.asarray(train_user_ids, dtype=np.int64)
valid_user_ids = np.asarray(valid_user_ids, dtype=np.int64)
test_user_ids = np.asarray(test_user_ids, dtype=np.int64)

print('train / valid / test 用户数 =', len(train_user_ids), len(valid_user_ids), len(test_user_ids))


train / valid / test 用户数 = 800 100 100



## 把 Junyi 映射成 AHS-KT 需要的字段

### question / concept
- `question_ids`：由 `exercise` 编号得到；
- `concept_ids`：由 `topic`（或 `area` / `exercise fallback`）编号得到。

### difficulty
AHS-KT 需要离散难度。这里采用一个简单但合理的近似：
- 在训练用户上统计正确率；
- 用 `1 - 正确率` 作为难度；
- 再离散到 `1..10`。

### behavior
- `attempts`：来自 `count_attempts`
- `hints`：来自 `count_hints`
- `speed`：用 `60 / time_taken` 近似每分钟完成题量，再做 `log1p`
- `behavior_cluster`：对 `(attempts, hints, speed)` 做聚类得到


In [8]:

question_names = sorted(df['exercise'].unique().tolist())
concept_names = sorted(df['concept_name'].unique().tolist())
question_to_id = {name: idx + 1 for idx, name in enumerate(question_names)}
concept_to_id = {name: idx + 1 for idx, name in enumerate(concept_names)}

df['question_ids'] = df['exercise'].map(question_to_id).astype(np.int32)
df['concept_ids'] = df['concept_name'].map(concept_to_id).astype(np.int32)
df['concept_ids_internal'] = (df['concept_ids'] - 1).astype(np.int32)

train_mask = df['user_id'].isin(train_user_ids)
train_df = df[train_mask].copy()
global_correct = float(train_df['correct'].mean())

def build_difficulty_map(frame, key_col):
    acc = frame.groupby(key_col)['correct'].mean()
    difficulty = np.clip(np.floor((1.0 - acc) * 10).astype(int) + 1, 1, 10)
    return difficulty.to_dict()

question_difficulty_map = build_difficulty_map(train_df, 'exercise')
concept_difficulty_map = build_difficulty_map(train_df, 'concept_name')
default_difficulty = int(np.clip(np.floor((1.0 - global_correct) * 10) + 1, 1, 10))

df['question_difficulty'] = df['exercise'].map(question_difficulty_map).fillna(default_difficulty).astype(np.int32)
df['concept_difficulty'] = df['concept_name'].map(concept_difficulty_map).fillna(default_difficulty).astype(np.int32)

print('全局训练正确率 =', round(global_correct, 4))
print('默认难度桶 =', default_difficulty)
df[['exercise', 'concept_name', 'question_ids', 'concept_ids', 'question_difficulty', 'concept_difficulty']].head()


全局训练正确率 = 0.8301
默认难度桶 = 2


,exercise,concept_name,question_ids,concept_ids,question_difficulty,concept_difficulty
0,multiplying_and_dividing_negative_numbers,absolute-value,326,1,2,3
1,division_4,multiplication-division,180,18,2,1
2,subtraction_2,addition-subtraction,498,2,1,1
3,matrix_basic_product,vectors-matrix,291,39,1,2
4,radius_and_diameter,basic-geometry,410,5,1,2


In [9]:

train_attempts_raw = train_df['count_attempts'].to_numpy(dtype=np.float32)
train_hints_raw = train_df['count_hints'].to_numpy(dtype=np.float32)
train_speed_raw = (60.0 / np.maximum(train_df['time_taken'].to_numpy(dtype=np.float32), 1.0)).astype(np.float32)

clip_values = {
    'attempts': float(np.quantile(train_attempts_raw, 0.99)),
    'hints': float(np.quantile(train_hints_raw, 0.99)),
    'speed': float(np.quantile(train_speed_raw, 0.99)),
}


def transform_behavior(frame):
    attempts = np.log1p(np.clip(frame['count_attempts'].to_numpy(dtype=np.float32), 0.0, clip_values['attempts'])).astype(np.float32)
    hints = np.log1p(np.clip(frame['count_hints'].to_numpy(dtype=np.float32), 0.0, clip_values['hints'])).astype(np.float32)
    speed_raw = (60.0 / np.maximum(frame['time_taken'].to_numpy(dtype=np.float32), 1.0)).astype(np.float32)
    speed = np.log1p(np.clip(speed_raw, 0.0, clip_values['speed'])).astype(np.float32)
    features = np.stack([attempts, hints, speed], axis=-1)
    return attempts, hints, speed, features

train_attempts, train_hints, train_speed, train_features = transform_behavior(train_df)
scaler = StandardScaler()
scaled_train_features = scaler.fit_transform(train_features)

cluster_model = MiniBatchKMeans(
    n_clusters=N_CLUSTERS,
    random_state=SEED,
    n_init=10,
    batch_size=4096,
)
cluster_model.fit(scaled_train_features)
raw_centers = scaler.inverse_transform(cluster_model.cluster_centers_)
cluster_order = sorted(
    range(len(raw_centers)),
    key=lambda i: (float(raw_centers[i, 0]), float(raw_centers[i, 1]), float(-raw_centers[i, 2])),
)
cluster_mapping = {int(old_label): int(new_label + 1) for new_label, old_label in enumerate(cluster_order)}

all_attempts, all_hints, all_speed, all_features = transform_behavior(df)
scaled_all_features = scaler.transform(all_features)
raw_cluster_labels = cluster_model.predict(scaled_all_features)
df['attempts'] = all_attempts
df['hints'] = all_hints
df['speed'] = all_speed
df['behavior_cluster'] = np.array([cluster_mapping[int(x)] for x in raw_cluster_labels], dtype=np.int32)

print('clip_values =', clip_values)
print('behavior_cluster 取值 =', sorted(df['behavior_cluster'].unique().tolist()))
df[['attempts', 'hints', 'speed', 'behavior_cluster']].head()


clip_values = {'attempts': 6.0, 'hints': 7.0, 'speed': 60.0}
behavior_cluster 取值 = [1, 2, 3, 4]


,attempts,hints,speed,behavior_cluster
0,0.693147,0.0,2.397895,2
1,0.693147,0.0,1.945910,1
2,1.386294,0.0,2.140066,4
3,0.693147,0.0,0.822681,1
4,0.693147,0.0,2.772589,2


In [10]:

df = df.sort_values(['user_id', 'timestamp']).reset_index(drop=True)

train_records = shuffle_records(build_sequence_records(df, train_user_ids, SEQUENCE_LENGTH), seed=2)
valid_records = shuffle_records(build_sequence_records(df, valid_user_ids, SEQUENCE_LENGTH), seed=2)
test_records = build_sequence_records(df, test_user_ids, SEQUENCE_LENGTH)

train_bundle = records_to_bundle(train_records, sequence_length=SEQUENCE_LENGTH)
valid_bundle = records_to_bundle(valid_records, sequence_length=SEQUENCE_LENGTH)
test_bundle = records_to_bundle(test_records, sequence_length=SEQUENCE_LENGTH)

print('train / valid / test 序列数 =', train_bundle.num_samples, valid_bundle.num_samples, test_bundle.num_samples)
print('sequence_length =', train_bundle.sequence_length)


train / valid / test 序列数 = 1707 221 208
sequence_length = 100


In [11]:

np.savez_compressed(TRAIN_NPZ_PATH, **train_bundle.as_dict())
np.savez_compressed(VALID_NPZ_PATH, **valid_bundle.as_dict())
np.savez_compressed(TEST_NPZ_PATH, **test_bundle.as_dict())

metadata = {
    'dataset_name': 'junyi',
    'subset_mode': 'all_eligible_users' if MAX_USERS is None else 'sampled_users_quickstart',
    'max_users': None if MAX_USERS is None else int(MAX_USERS),
    'min_interactions': int(MIN_INTERACTIONS),
    'sequence_length': int(SEQUENCE_LENGTH),
    'num_questions': int(df['question_ids'].max()),
    'num_concepts': int(df['concept_ids'].max()),
    'num_question_difficulty': 10,
    'num_concept_difficulty': 10,
    'num_behavior_clusters': int(N_CLUSTERS + 1),
    'num_filtered_interactions': int(len(df)),
    'num_users': int(df['user_id'].nunique()),
    'global_correct_train': float(global_correct),
    'clip_values': {key: float(value) for key, value in clip_values.items()},
    'split_summary': {
        'train_users': int(len(train_user_ids)),
        'valid_users': int(len(valid_user_ids)),
        'test_users': int(len(test_user_ids)),
        'train_sequences': int(train_bundle.num_samples),
        'valid_sequences': int(valid_bundle.num_samples),
        'test_sequences': int(test_bundle.num_samples),
    },
}
METADATA_PATH.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')

config_payload = {
    'project_name': 'ahs-kt',
    'task_name': TASK_NAME,
    'seed': int(SEED),
    'dataset': {
        'mode': 'real_npz',
        'train_path': str(TRAIN_NPZ_PATH.relative_to(PROJECT_ROOT)),
        'valid_path': str(VALID_NPZ_PATH.relative_to(PROJECT_ROOT)),
        'test_path': str(TEST_NPZ_PATH.relative_to(PROJECT_ROOT)),
    },
    'model': {
        'num_questions': int(df['question_ids'].max()),
        'num_concepts': int(df['concept_ids'].max()),
        'num_question_difficulty': 10,
        'num_concept_difficulty': 10,
        'num_behavior_clusters': int(N_CLUSTERS + 1),
        'sequence_length': int(SEQUENCE_LENGTH),
        'embedding_dim': 64,
        'difficulty_dim': 32,
        'behavior_dim': 32,
        'hidden_dim': 96,
        'dropout': 0.2,
        'use_behavior_cluster': True,
        'use_difficulty_features': True,
        'use_behavior_features': True,
        'use_target_interaction': False,
        'fusion_mode': 'early',
        'behavior_condition_on_difficulty': True,
        'aux_residual_scale': 0.25,
        'difficulty_mode': 'embedding',
        'difficulty_bias_scale': 0.1,
        'difficulty_feature_source': 'question_concept',
        'question_global_easiness': float(global_correct),
        'concept_global_easiness': float(global_correct),
    },
    'training': {
        'epochs': int(EPOCHS),
        'batch_size': int(BATCH_SIZE),
        'learning_rate': float(LEARNING_RATE),
        'patience': int(PATIENCE),
    },
    'demo': {
        'train_size': 0,
        'valid_size': 0,
        'test_size': 0,
    },
    'outputs': {
        'root_dir': str(OUTPUT_DIR.relative_to(PROJECT_ROOT)),
    },
}
CONFIG_PATH.write_text(json.dumps(config_payload, ensure_ascii=False, indent=2), encoding='utf-8')

print('saved:', TRAIN_NPZ_PATH)
print('saved:', VALID_NPZ_PATH)
print('saved:', TEST_NPZ_PATH)
print('saved:', METADATA_PATH)
print('saved:', CONFIG_PATH)


saved: /root/autodl-tmp/ahs-kt/data/junyi/junyi_quick_train_ahskt.npz
saved: /root/autodl-tmp/ahs-kt/data/junyi/junyi_quick_valid_ahskt.npz
saved: /root/autodl-tmp/ahs-kt/data/junyi/junyi_quick_test_ahskt.npz
saved: /root/autodl-tmp/ahs-kt/data/junyi/junyi_quick_metadata.json
saved: /root/autodl-tmp/ahs-kt/configs/ahskt_junyi_quick.json



## 使用项目原生训练代码开始训练

下面这段代码和 `scripts/train_ahskt.py` 的核心逻辑保持一致：
- 读取 config
- 加载 bundle
- 初始化 `AHSKTModel`
- 调 `fit_and_evaluate`
- 输出 metrics JSON


In [12]:

config = load_config(CONFIG_PATH, project_root=PROJECT_ROOT)

from ahskt.data.dataset import load_bundle_from_config

train_bundle_loaded, valid_bundle_loaded, test_bundle_loaded = load_bundle_from_config(config)
model = AHSKTModel(config.model)

train_start = time.time()
metrics_summary = fit_and_evaluate(
    model=model,
    train_bundle=train_bundle_loaded,
    valid_bundle=valid_bundle_loaded,
    test_bundle=test_bundle_loaded,
    config=config,
)
train_elapsed = time.time() - train_start

metrics_path = config.output_root / f'{config.task_name}_metrics.json'
metrics_path.write_text(json.dumps(metrics_summary, ensure_ascii=False, indent=2), encoding='utf-8')

print('训练耗时(秒) =', round(train_elapsed, 2))
print('metrics_path =', metrics_path)
print(json.dumps(metrics_summary['test_metrics'], ensure_ascii=False, indent=2))


2026-04-07 10:46:28.520740: I tensorflow/core/platform/cpu_feature_guard.cc:151] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-07 10:46:29.210346: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1525] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22182 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:57:00.0, compute capability: 8.6
2026-04-07 10:46:29.901171: I tensorflow/stream_executor/cuda/cuda_blas.cc:1786] TensorFloat-32 will be used for the matrix multiplication. This will only be logged once.
2026-04-07 10:46:30.395415: I tensorflow/stream_executor/cuda/cuda_dnn.cc:368] Loaded cuDNN version 8200


训练耗时(秒) = 4.95
metrics_path = /root/autodl-tmp/ahs-kt/outputs/junyi_quick_run/ahskt_junyi_quickstart_1k_metrics.json
{
  "loss": 0.36662188172340393,
  "auc": 0.7410875801679588,
  "acc": 0.862877234104893,
  "rmse": 0.331177294254303
}



## 额外补一项：F1

项目原生 `metrics.py` 默认只输出：
- `loss`
- `auc`
- `acc`
- `rmse`

这里我们额外在 Notebook 里基于测试集预测结果计算一个 `f1`，方便做更完整的 quickstart 结果汇报。


In [13]:

def collect_targets_and_predictions(model, bundle, batch_size):
    dataset = bundle.to_tf_dataset(batch_size=batch_size, shuffle=False)
    all_targets = []
    all_predictions = []
    for batch in dataset:
        logits = model(batch, training=False)
        next_logits = logits[:, :-1]
        next_targets = tf.cast(batch['responses'][:, 1:], tf.float32)
        next_mask = tf.cast(batch['mask'][:, 1:], tf.float32)
        valid_logits = tf.boolean_mask(next_logits, next_mask > 0)
        valid_targets = tf.boolean_mask(next_targets, next_mask > 0)
        all_targets.append(valid_targets.numpy())
        all_predictions.append(tf.sigmoid(valid_logits).numpy())
    return np.concatenate(all_targets, axis=0), np.concatenate(all_predictions, axis=0)


test_targets, test_predictions = collect_targets_and_predictions(
    model=model,
    bundle=test_bundle_loaded,
    batch_size=config.training.batch_size,
)

test_binary_predictions = (test_predictions > 0.5).astype(int)
summary_with_f1 = {
    'acc': float(accuracy_score(test_targets, test_binary_predictions)),
    'auc': float(roc_auc_score(test_targets, test_predictions)),
    'f1': float(f1_score(test_targets, test_binary_predictions)),
    'loss': float(metrics_summary['test_metrics']['loss']),
    'rmse': float(metrics_summary['test_metrics']['rmse']),
    'num_test_points': int(len(test_targets)),
}

summary_with_f1_path = config.output_root / f'{config.task_name}_metrics_with_f1.json'
summary_with_f1_path.write_text(json.dumps(summary_with_f1, ensure_ascii=False, indent=2), encoding='utf-8')

print('summary_with_f1_path =', summary_with_f1_path)
print(json.dumps(summary_with_f1, ensure_ascii=False, indent=2))


summary_with_f1_path = /root/autodl-tmp/ahs-kt/outputs/junyi_quick_run/ahskt_junyi_quickstart_1k_metrics_with_f1.json
{
  "acc": 0.862877234104893,
  "auc": 0.7410875801679588,
  "f1": 0.9241183623834617,
  "loss": 0.36662188172340393,
  "rmse": 0.331177294254303,
  "num_test_points": 13652
}



## 想跑更正式一点怎么改

如果你后面想把这个 quickstart 扩成更正式的 Junyi 实验，可以按这个顺序改：

1. 把 `MAX_USERS = 1000` 改大，或者设成 `None`
2. 把 `EPOCHS = 2` 提高到 `5~20`
3. 调整模型维度和 `patience`
4. 固定多个随机种子做重复实验
5. 在 `topic` 之外尝试更细的 concept 定义

这本 Notebook 的重点是：**先把 AHS-KT + Junyi 原始数据这条链路跑通。**
